In [ ]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import pandas as pd
import polars as pl
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time
from sklearn.cluster import DBSCAN, HDBSCAN, OPTICS
from sklearn.neighbors import KDTree
from sklearn.neighbors import NearestNeighbors

import sys

sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter(dark_background=True)

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions()

from src import SR_Functions

SRes_F = SR_Functions.SuperRes_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

from src import render
from src import localize as _localize
from src import postprocess as _postprocess

import types

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
# data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = "/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/Ximea_Calibration/"
# data_folder = r'C:\Users\jsb92\Cambridge University Dropbox\Joseph Beckwith\Chemistry\Lee\Data\Salix\Ximea_Calibration'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()

In [ ]:
data_folders = np.array(
    [
        r"/media/jbeckwith/Ezra Seagat/JSB/20250520_DNAOrigami/data/15mW/",
        r"/media/jbeckwith/Ezra Seagat/JSB/20250520_DNAOrigami/data/30mW/",
    ]
)

In [ ]:
example_folder = data_folders[0]

In [ ]:
localisation_files = H_F.file_search(example_folder, ".h5", "")

In [ ]:
metadata_files = H_F.file_search(example_folder, "metadata", "")

In [ ]:
localisation_files

In [ ]:
columns = [
    "xc",
    "yc",
    "s_x",
    "s_y",
    "bg_B",
    "bg_G",
    "bg_R",
    "A_B",
    "A_G",
    "A_R",
    "chi_sqr",
    "frame",
    "photons",
    "xc_err",
    "yc_err",
    "s_x_err",
    "s_y_err",
    "bg_B_err",
    "bg_G_err",
    "bg_R_err",
    "A_B_err",
    "A_G_err",
    "A_R_err",
]

loc_data = pd.read_hdf(localisation_files[2])

In [ ]:
x_coord, y_coord, width, height = IO.metadata_reader_imageJ(metadata_files[0])
n_frames = IO.metadata_nframes_reader_imageJ(metadata_files[0])

In [ ]:
loc_data["photons"] = loc_data["A_B"] + loc_data["A_G"] + loc_data["A_R"]
loc_data = loc_data[loc_data["xc_err"] < 1]
loc_data = loc_data[loc_data["yc_err"] < 1]

loc_data = loc_data[loc_data["xc_err"] > 0]
loc_data = loc_data[loc_data["yc_err"] > 0]

loc_data = loc_data[loc_data["s_x_err"] < 1]
loc_data = loc_data[loc_data["s_y_err"] < 1]

loc_data = loc_data[loc_data["A_B_err"] < 100]
loc_data = loc_data[loc_data["A_G_err"] < 100]
loc_data = loc_data[loc_data["A_R_err"] < 100]

loc_data = loc_data[loc_data["bg_B_err"] < 3]
loc_data = loc_data[loc_data["bg_G_err"] < 100]
loc_data = loc_data[loc_data["bg_R_err"] < 100]

loc_data = loc_data[loc_data["photons"] < 10000]
# loc_data = loc_data[loc_data['A_B']+loc_data['A_G']+loc_data['A_R'] > 500]
loc_data = loc_data[loc_data["A_B"] > 0]
loc_data = loc_data[loc_data["A_G"] > 0]
loc_data = loc_data[loc_data["A_R"] > 0]
loc_data["A_B"] = loc_data["A_B"] / loc_data["photons"]
loc_data["A_G"] = loc_data["A_G"] / loc_data["photons"]
loc_data["A_R"] = loc_data["A_R"] / loc_data["photons"]


loc_data = loc_data[loc_data["chi_sqr"] < 2]

In [ ]:
np.arange(0, n_frames + 1, 10000)

In [ ]:
def iterative_construction_drift(loc_data, n_frames_per=10000, n_frames_total=200000):
    frame_subsets = np.arange(0, n_frames + 1, 10000)
    for i, low_frame in enumerate(frame_subsets[:-1]):
        high_frame = frame_subsets[i + 1]
        subset_loc = loc_data[low_frame < loc_data["frame"] <= high_frame]
        hist = np.histogram(
            image.flatten(), bins=np.histogram_bin_edges(image.flatten(), "sturges")
        )
        threshold = np.percentile(hist[0], 95)
        pixelsize = 69
        box = int(np.round(900 / pixelsize))
        box = box + 1 if box % 2 == 0 else box
        y, x, _ = _localize.identify_in_image(image, threshold, box=box)
        picks = [(xi, yi) for xi, yi in zip(x, y)]
        # select the picks with appropriate number of localizations
        min_n = 0.9 * n_frames_per
        picked_locs = _postprocess.picked_locs(
            subset_loc.to_records(index=False),
            width,
            height,
            picks,
            "Circle",
            pick_size=box / 2,
            add_group=False,
        )
        picks = [pick for i, pick in enumerate(picks) if len(picked_locs[i]) > min_n]
    return picked_locs

In [ ]:
image = render.render(
    locs=loc_data.to_records(index=False),
    oversampling=1,
    viewport=((0, 0), (height, width)),
    blur_method="smooth",
)[1]

In [ ]:
hist = np.histogram(
    image.flatten(), bins=np.histogram_bin_edges(image.flatten(), "sturges")
)
threshold = np.percentile(hist[0], 80)
pixelsize = 69
box = int(np.round(1250 / pixelsize))
box = box + 1 if box % 2 == 0 else box
y, x, _ = _localize.identify_in_image(image, threshold, box=box)
picks = [(xi, yi) for xi, yi in zip(x, y)]
# select the picks with appropriate number of localizations
n_frames = 200000
min_n = 0.8 * n_frames
temp_picked_locs = _postprocess.picked_locs(
    loc_data.to_records(index=False),
    width,
    height,
    picks,
    "Circle",
    pick_size=box / 2,
    add_group=False,
)
picks = [pick for i, pick in enumerate(picks) if len(temp_picked_locs[i]) > min_n]
picked_locs = _postprocess.picked_locs(
    loc_data.to_records(index=False),
    width,
    height,
    picks,
    "Circle",
    pick_size=box / 2,
    add_group=False,
)

In [ ]:
plt.scatter(
    loc_data["xc"][0:10000], loc_data["yc"][0:10000], s=0.5, alpha=0.1, color="darkblue"
)
for pick in picked_locs:
    plt.scatter(pick["xc"][0], pick["yc"][0], s=5, alpha=1, color="red")
    plt.scatter(pick["xc"][-1], pick["yc"][-1], s=5, alpha=1, color="red")

In [ ]:
X = np.vstack([loc_data["xc"], loc_data["yc"]]).T
loc_precision = 0.5 * (np.mean(loc_data["xc_err"]) + np.mean(loc_data["yc_err"]))

In [ ]:
hdb = OPTICS(max_eps=loc_precision, min_samples=100, cluster_method="dbscan")
hdb.fit(X)

In [ ]:
loc_data_nofiducials = loc_data[hdb.labels_ < 0]

In [ ]:
plt.scatter(
    loc_data_nofiducials["xc"][0:10000],
    loc_data_nofiducials["yc"][0:10000],
    s=0.5,
    alpha=0.1,
    color="darkblue",
)

In [ ]:
image = render.render(
    locs=loc_data.to_records(index=False),
    oversampling=8,
    viewport=((0, 0), (height, width)),
    blur_method="gaussian",
)[1]

In [ ]:
drift = _postprocess.undrift_from_picked(picked_locs, n_frames)

In [ ]:
def undrift(drift, loc_data, width, height):
    drift_x = {}
    drift_y = {}
    test_keys = np.arange(n_frames)

    for key in test_keys:
        drift_x[key] = drift["xc"][key]
        drift_y[key] = drift["yc"][key]
    loc_data["xc"] = loc_data["xc"] - loc_data["frame"].map(drift_x)
    loc_data["yc"] = loc_data["yc"] - loc_data["frame"].map(drift_y)

    loc_data = loc_data[loc_data["xc"] < width]
    loc_data = loc_data[loc_data["yc"] < height]
    loc_data = loc_data.reindex()
    return loc_data

In [ ]:
plt.plot(drift["xc"] * 69)
plt.plot(drift["yc"] * 69)
# plt.xlim([0, 25000])

In [ ]:
undrifted_loc_data = undrift(drift, loc_data)

In [ ]:
undrifted_loc_data = undrifted_loc_data.drop("level_0")

In [ ]:
plt.scatter(
    undrifted_loc_data["xc"][0:10000],
    undrifted_loc_data["yc"][0:10000],
    s=0.5,
    alpha=0.1,
    color="darkblue",
)

In [ ]:
IO._write_h5_database(
    loc_data,
    "/media/jbeckwith/Ezra Seagat/JSB/20250520_DNAOrigami/data/15mW/Localisations_Pruned_Undrifted.h5",
)

In [ ]:
plt.hist(undrifted_loc_data["A_B"], 400, density=True, color="blue")
plt.hist(undrifted_loc_data["A_R"], 400, density=True, color="red")
plt.hist(undrifted_loc_data["A_G"], 400, density=True, color="green")
plt.xlim([0, 1])
plt.ylim([0, 10])
# plt.yscale('log')
plt.show()

In [ ]:
overall_drift = (drift["xc"] * 69 + drift["yc"] * 69) / 2

In [ ]:
loc_data = loc_data[loc_data["frame"] > 10000]

In [ ]:
drift_x = {}
drift_y = {}
test_keys = np.arange(n_frames)

for key in test_keys:
    drift_x[key] = drift["xc"][key]
    drift_y[key] = drift["yc"][key]
loc_data["xc"] = loc_data["xc"] - loc_data["frame"].map(drift_x)
loc_data["yc"] = loc_data["yc"] - loc_data["frame"].map(drift_y)

In [ ]:
columns = [
    "xc",
    "yc",
    "s_x",
    "s_y",
    "bg_B",
    "bg_G",
    "bg_R",
    "A_B",
    "A_G",
    "A_R",
    "chi_sqr",
    "frame",
    "xc_err",
    "yc_err",
    "s_x_err",
    "s_y_err",
    "bg_B_err",
    "bg_G_err",
    "bg_R_err",
    "A_B_err",
    "A_G_err",
    "A_R_err",
]


loc_data = pd.read_hdf(localisation_files[0])

In [ ]:
np.std(loc_data["bg_B_err"])

In [ ]:
plt.hist(loc_data["bg_R_err"], 1000)
plt.show()

In [ ]:
plt.hist(loc_data["A_B"], 400, density=True)
plt.hist(loc_data["A_R"], 400, density=True)
plt.hist(loc_data["A_G"], 400, density=True)
plt.xlim([0, 1])
plt.ylim([0, 10])
# plt.yscale('log')
plt.show()

In [ ]:
min_n = 0.9 * n_frames

In [ ]:
min_n

In [ ]:
plt.scatter(loc_data["xc"], loc_data["yc"], s=0.1, alpha=0.01, color="darkblue")
for pick in picked_locs:
    plt.scatter(pick["xc"], pick["yc"], s=0.1, alpha=1, color="red")

In [ ]:
drift = _postprocess.undrift_from_picked(picked_locs, n_frames)

In [ ]:
drift_x = {}
drift_y = {}
test_keys = np.arange(n_frames)

for key in test_keys:
    drift_x[key] = drift["xc"][key]
    drift_y[key] = drift["yc"][key]

In [ ]:
loc_data_undrifted["xc"] = loc_data_undrifted["xc"] - loc_data_undrifted["frame"].map(
    drift_x
)
loc_data_undrifted["yc"] = loc_data_undrifted["yc"] - loc_data_undrifted["frame"].map(
    drift_y
)

In [ ]:
plt.scatter(
    loc_data_undrifted["xc"],
    loc_data_undrifted["yc"],
    s=0.1,
    alpha=0.01,
    color="darkblue",
)
plt.xlim([10, 20])
plt.ylim([115, 118])
plt.show()

In [ ]:
IO._write_h5_database(
    loc_data_undrifted,
    "/media/jbeckwith/Ezra Seagat/JSB/20250520_DNAOrigami/data/15mW/Localisations_UnDrifted.h5",
)

In [ ]:
X = np.vstack([loc_data["xc"], loc_data["yc"]]).T
loc_precision = 0.5 * (np.mean(loc_data["xc_err"]) + np.mean(loc_data["yc_err"]))
hdb = DBSCAN(eps=loc_precision, min_samples=int(0.8 * n_frames))
hdb.fit(X)

In [ ]:
test2 = loc_data_undrifted[hdb.labels_ < 0]

In [ ]:
from src import render

oversampling = 2
locs, rendered_object = render.render(
    loc_data_undrifted.to_records(index=False),
    viewport=((0, 0), (width, height)),
    blur_method="gaussian",
    oversampling=oversampling,
)

In [ ]:
locs

In [ ]:
fig, axs = plotter.two_column_plot()

axs = plotter.image_plot(
    axs=axs,
    data=rendered_object.T,
    vmin=np.percentile(rendered_object, 0.1),
    vmax=np.percentile(rendered_object, 99),
    cbar="on",
    cbarlabel="localisations",
    pixelsize=69 / oversampling,
    cmap="inferno",
)
plt.show()